<a href="https://colab.research.google.com/github/giankev/lightweight-model-for-signal-processing/blob/ofdm-case/dataset_creation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
pip install sionna tensorflow

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 520.4/520.4 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.4/8.4 MB 61.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.0/5.0 MB 74.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.0/63.0 MB 9.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 63.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.8/139.8 kB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 71.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 271.7/271.7 kB 10.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 53.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 53.2 MB/s eta 0:00:00
  Attempting uninstall: widgetsnbextension
    Found existing installation: widgetsnbextension 3.6.10
    Uninstalling widgetsnbextension

# Import

In [7]:
# ============================================================
# NOTEBOOK: Generate the 4 OFDM datasets used in the project
#
# Output files:
#   1) ofdm_standard_train.npz
#   2) ofdm_standard_test.npz
#   3) ofdm_starved_train.npz
#   4) ofdm_starved_test.npz
#
# ============================================================

# =========================
# 1) Imports
# =========================
import os
import random
import numpy as np
import tensorflow as tf
import torch

from sionna_dg import SimConfig, generate_dataset_offline


# =========================
# 2) Reproducibility
# =========================
def set_seed_all(seed: int = 46):
    random.seed(seed)
    np.random.seed(seed)
    tf.random.set_seed(int(seed))
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


# =========================
# 3) Dataset builders
# =========================
def preprocess_dataset_for_training(base_cfg: SimConfig, ds: dict) -> dict:
    """
    Convert the offline dataset produced by generate_dataset_offline(...)
    """
    n_pilot_sym = base_cfg.ofdm_num_pilot_symbols

    x_rx = ds["rx_iq_data_time"]          # [N, T_data, 2]
    x_pilot = ds["rx_iq_pilot_time"]      # [N, T_pilot, 2]
    coded_bits = ds["coded_bits"]         # [N, n_bits]
    tx_grid = ds["tx_grid"]               # [N, n_total_sym, fft, 2]

    # Keep only data OFDM symbols from TX grid, then flatten over OFDM symbols
    x_data = tx_grid[:, n_pilot_sym:, :, :].reshape(tx_grid.shape[0], -1, 2)

    # Match the exact training layout
    x_rx_np = np.transpose(x_rx.astype(np.float32), (0, 2, 1))                # [N, 2, T_data]
    x_pilot_np = np.transpose(x_pilot.astype(np.float32), (0, 2, 1))          # [N, 2, T_pilot]
    target_bits_np = coded_bits.reshape(-1, base_cfg.seq_length, 4).transpose(0, 2, 1).astype(np.float32)  # [N, 4, seq_length]
    target_iq_np = np.transpose(x_data.astype(np.float32), (0, 2, 1))         # [N, 2, seq_length]

    phi0_np = ds["phi0"].astype(np.float32).squeeze(-1)                        # [N]
    cfo_np = ds["cfo_norm"].astype(np.float32).squeeze(-1)                     # [N]
    ebn0_db_np = ds["ebn0_db"].astype(np.float32)                              # [N]
    no_np = ds["no"].astype(np.float32).squeeze(-1)                            # [N]
    info_bits_np = ds["info_bits"].astype(np.uint8)                            # [N, k]
    coded_bits_flat_np = ds["coded_bits"].astype(np.uint8)                     # [N, n]

    return {
        "x_rx": x_rx_np,
        "x_pilot": x_pilot_np,
        "target_bits": target_bits_np,
        "target_iq": target_iq_np,
        "phi0": phi0_np,
        "cfo": cfo_np,
        "ebn0_db": ebn0_db_np,
        "no": no_np,
        "info_bits": info_bits_np,
        "coded_bits_flat": coded_bits_flat_np,
    }


def build_mixed_training_dataset(base_cfg: SimConfig,
                                 main_examples: int = 8000,
                                 main_ebn0=(4.0, 8.0),
                                 rand_examples: int = 2000,
                                 rand_ebn0=(0.0, 12.0)) -> dict:
    """
    Build the exact mixed training dataset used in the notebooks:
      - 80% focused on [4, 8] dB
      - 20% generalized on [0, 12] dB
    """
    print(f"Generating mixed training dataset for waveform={base_cfg.waveform}")
    print(f"  Main subset : {main_examples} examples in [{main_ebn0[0]}, {main_ebn0[1]}] dB")
    print(f"  Rand subset : {rand_examples} examples in [{rand_ebn0[0]}, {rand_ebn0[1]}] dB")

    cfg_main = SimConfig(**{
        **base_cfg.__dict__,
        "num_examples": int(main_examples),
        "ebn0_db_min": float(main_ebn0[0]),
        "ebn0_db_max": float(main_ebn0[1]),
    })

    cfg_rand = SimConfig(**{
        **base_cfg.__dict__,
        "num_examples": int(rand_examples),
        "ebn0_db_min": float(rand_ebn0[0]),
        "ebn0_db_max": float(rand_ebn0[1]),
    })

    ds_main = generate_dataset_offline(cfg_main)
    ds_rand = generate_dataset_offline(cfg_rand)

    proc_main = preprocess_dataset_for_training(base_cfg, ds_main)
    proc_rand = preprocess_dataset_for_training(base_cfg, ds_rand)

    mixed = {}
    for key in proc_main.keys():
        mixed[key] = np.concatenate([proc_main[key], proc_rand[key]], axis=0)

    return mixed


def build_direct_dataset(cfg: SimConfig) -> dict:
    """
    Build a direct dataset (used for test sets).
    """
    print(f"Generating direct dataset for waveform={cfg.waveform}")
    print(f"  Examples : {cfg.num_examples}")
    print(f"  Eb/N0    : [{cfg.ebn0_db_min}, {cfg.ebn0_db_max}] dB")
    print(f"  Seed     : {cfg.seed}")
    print(f"  SeedNoise: {cfg.seed_noise}")
    ds = generate_dataset_offline(cfg)
    return preprocess_dataset_for_training(cfg, ds)


def build_train_val_indices(num_samples: int, train_ratio: float = 0.85, seed: int = 46):
    """
    Deterministic split indices for training datasets.
    """
    g = torch.Generator().manual_seed(seed)
    perm = torch.randperm(num_samples, generator=g).numpy()

    train_size = int(train_ratio * num_samples)
    train_idx = perm[:train_size]
    val_idx = perm[train_size:]

    return train_idx.astype(np.int64), val_idx.astype(np.int64)


def save_training_npz(path: str, data: dict, split_seed: int = 46, train_ratio: float = 0.85):
    train_idx, val_idx = build_train_val_indices(
        num_samples=data["x_rx"].shape[0],
        train_ratio=train_ratio,
        seed=split_seed,
    )

    np.savez_compressed(
        path,
        x_rx=data["x_rx"],
        x_pilot=data["x_pilot"],
        target_bits=data["target_bits"],
        target_iq=data["target_iq"],
        phi0=data["phi0"],
        cfo=data["cfo"],
        ebn0_db=data["ebn0_db"],
        no=data["no"],
        info_bits=data["info_bits"],
        coded_bits_flat=data["coded_bits_flat"],
        train_idx=train_idx,
        val_idx=val_idx,
    )


def save_test_npz(path: str, data: dict):
    np.savez_compressed(
        path,
        x_rx=data["x_rx"],
        x_pilot=data["x_pilot"],
        target_bits=data["target_bits"],
        target_iq=data["target_iq"],
        phi0=data["phi0"],
        cfo=data["cfo"],
        ebn0_db=data["ebn0_db"],
        no=data["no"],
        info_bits=data["info_bits"],
        coded_bits_flat=data["coded_bits_flat"],
    )


def sanity_check_npz(path: str):
    pack = np.load(path)
    print(f"\nSanity check: {os.path.basename(path)}")
    print("-" * 70)
    for key in pack.files:
        print(f"{key:16s} shape={str(pack[key].shape):20s} dtype={pack[key].dtype}")
    print("-" * 70)


# =========================
# 5) Configurations
# =========================

# -------------------------
# Standard training config
# -------------------------
cfg_ofdm_standard = SimConfig(
    num_examples=10000,
    bits_per_symbol=4,
    k=256,
    demap_method="app",
    dec_num_iter=15,
    cn_update="minsum",
    seed=46,
    channels_last=True,
    cfo_mode="direct",
    cfo_norm_min=-2e-4,
    cfo_norm_max=+2e-4,
    ebn0_db_min=0.0,
    ebn0_db_max=12.0,
    include_pilot_overhead_in_no=True,
    waveform="ofdm",
    ofdm_fft_size=64,
    ofdm_cp_len=16,
    ofdm_num_data_symbols=2,
    ofdm_num_pilot_symbols=1,
)

# -------------------------
# Standard test config
# -------------------------
cfg_ofdm_standard_test = SimConfig(**{
    **cfg_ofdm_standard.__dict__,
    "num_examples": 5000,
    "seed": 999,
    "seed_noise": 888,
})

# -------------------------
# Starved training config
# -------------------------
cfg_ofdm_starved = SimConfig(
    num_examples=10000,
    bits_per_symbol=4,
    k=1024,
    seq_length=512,
    demap_method="app",
    dec_num_iter=15,
    cn_update="minsum",
    seed=46,
    channels_last=True,
    cfo_mode="direct",
    cfo_norm_min=-2e-4,
    cfo_norm_max=+2e-4,
    ebn0_db_min=0.0,
    ebn0_db_max=12.0,
    include_pilot_overhead_in_no=True,
    waveform="ofdm",
    ofdm_fft_size=64,
    ofdm_cp_len=16,
    ofdm_num_data_symbols=8,
    ofdm_num_pilot_symbols=1,
)

# -------------------------
# Starved test config
# -------------------------
NUM_EVAL_EXAMPLES = 5000

cfg_ofdm_starved_test = SimConfig(**{
    **cfg_ofdm_starved.__dict__,
    "num_examples": NUM_EVAL_EXAMPLES,
    "seed": 999,
    "seed_noise": 888,
})


# =========================
# 6) Main generation
# =========================
if __name__ == "__main__":
    save_dir = "./dataset_export"
    os.makedirs(save_dir, exist_ok=True)

    # ---- 1) Standard training dataset
    set_seed_all(cfg_ofdm_standard.seed)
    ds_standard_train = build_mixed_training_dataset(cfg_ofdm_standard)
    path_standard_train = os.path.join(save_dir, "ofdm_standard_train.npz")
    save_training_npz(path_standard_train, ds_standard_train, split_seed=cfg_ofdm_standard.seed, train_ratio=0.85)
    print(f"\nSaved: {path_standard_train}")
    sanity_check_npz(path_standard_train)

    # ---- 2) Standard test dataset
    set_seed_all(cfg_ofdm_standard_test.seed)
    ds_standard_test = build_direct_dataset(cfg_ofdm_standard_test)
    path_standard_test = os.path.join(save_dir, "ofdm_standard_test.npz")
    save_test_npz(path_standard_test, ds_standard_test)
    print(f"\nSaved: {path_standard_test}")
    sanity_check_npz(path_standard_test)

    # ---- 3) Starved training dataset
    set_seed_all(cfg_ofdm_starved.seed)
    ds_starved_train = build_mixed_training_dataset(cfg_ofdm_starved)
    path_starved_train = os.path.join(save_dir, "ofdm_starved_train.npz")
    save_training_npz(path_starved_train, ds_starved_train, split_seed=cfg_ofdm_starved.seed, train_ratio=0.85)
    print(f"\nSaved: {path_starved_train}")
    sanity_check_npz(path_starved_train)

    # ---- 4) Starved test dataset
    set_seed_all(cfg_ofdm_starved_test.seed)
    ds_starved_test = build_direct_dataset(cfg_ofdm_starved_test)
    path_starved_test = os.path.join(save_dir, "ofdm_starved_test.npz")
    save_test_npz(path_starved_test, ds_starved_test)
    print(f"\nSaved: {path_starved_test}")
    sanity_check_npz(path_starved_test)

    print("\nAll 4 datasets generated")

Generating mixed training dataset for waveform=ofdm
  Main subset : 8000 examples in [4.0, 8.0] dB
  Rand subset : 2000 examples in [0.0, 12.0] dB

Saved: ./dataset_export/ofdm_standard_train.npz

Sanity check: ofdm_standard_train.npz
----------------------------------------------------------------------
x_rx             shape=(10000, 2, 160)      dtype=float32
x_pilot          shape=(10000, 2, 80)       dtype=float32
target_bits      shape=(10000, 4, 128)      dtype=float32
target_iq        shape=(10000, 2, 128)      dtype=float32
phi0             shape=(10000,)             dtype=float32
cfo              shape=(10000,)             dtype=float32
ebn0_db          shape=(10000,)             dtype=float32
no               shape=(10000,)             dtype=float32
info_bits        shape=(10000, 256)         dtype=uint8
coded_bits_flat  shape=(10000, 512)         dtype=uint8
train_idx        shape=(8500,)              dtype=int64
val_idx          shape=(1500,)              dtype=int64
------